# Estimate agricultural land fraction using CORINE land cover (exposure)

This short how-to guides you through the steps to create a timeseries of the agricultural land fraction using CORINE land cover data and save it as a csv file. The final csv file can be used as, e.g., exposure data for the drought risk estimation. 

## Manual download of CORINE land cover data via web interface

We download the CORINE land cover manually using the Copernicus Land viewer available through: [https://land.copernicus.eu/en/products/corine-land-cover](https://land.copernicus.eu/en/products/corine-land-cover)

In the viewer, we select the CORINE land cover data for all years available, i.e., 1990, 2000, 2006, 2012, and 2018. To reduce the amount of data that we download, we further reduce the spatial extent to the administrative region that we're looking at, e.g., the NUTS-2 region "Central Greece" (EL64). The image below shows the selection process. 

![Image](../img/corine_download_viewer.png "CORINE land cover data selected for this example: NUTS region 2 'EL64' (Central Greece), and CORINE land cover from 1990, 2000, 2006, 2012, and 2018.")

Once selected, we add click on the download symbol for each dataset (year), and process to the cart.  To facilitate the use of this data for our purposes, we select the "raster" format and download the data on the EPSG: 4326 projection; a projection typically used for climate projections. The image below shows the selection of format and projection. 
![Image](../img/corine_download_cart.png "CORINE land cover download.")
 

In [ ]:
Once downloaded, we move this data into the following subdirectory: 

In [ ]:
"./data/corine_landcover"

## Read and visualize CORINE land cover data

In [ ]:
import zipfile
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import regionmask
from pathlib import Path
data_dir = Path("../../data/corine_landcover")

In [ ]:
# List all tif-files in data_dir
tif_files = list(data_dir.glob("*.tif"))
print(f"Found {len(tif_files)} tif files.")
print(tif_files)

In [ ]:
# Read all tif files into a single xarray dataset
datasets = []
for tif_file in tif_files:
    ds = xr.open_rasterio(tif_file)
    datasets.append(ds) 
# concatenate along new time dimension  
corine_ds = xr.concat(datasets, dim="time")
corine_ds = corine_ds.rename({"band": "land_cover", "x": "lon", "y": "lat"})
# assign time based on filename of tif_files (extract YYYY from filename)
corine_ds["time"] = pd.date_range(start="1990", periods=len(tif_files), freq="5Y")
corine_ds = corine_ds.assign_coords(time=corine_ds.time)
corine_ds = corine_ds.squeeze("land_cover")  # remove land_cover dimension  
corine_ds.load()